In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

## Buffer Memory
* ConversationBufferMemory keeps a list of chat messages in a buffer and passes those into the prompt template.

In [4]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. Define the Prompt Template (Unchanged, but modernized imports)
prompt = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template(
            "You are a nice chatbot having a conversation with a human."
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

# 2. Build the Core LCEL Chain
# (Assuming 'llm' is already initialized, e.g., ChatOpenAI, ChatAnthropic, etc.)
base_chain = prompt | llm

# 3. Manage Session Memories Globally
# In modern LangChain, an dictionary stores history for different users/sessions.
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 4. Wrap the base chain with message history tracking
conversation = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)


C:\Users\DulamVinayGopal\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [5]:
# First interaction
response1 = conversation.invoke(
    {"question": "Hi! My name is Vinay."},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response1.content)

# Second interaction (It will remember your name!)
response2 = conversation.invoke(
    {"question": "What is my name?"},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response2.content)


Hello Vinay! It's nice to meet you. How are you doing today?
Your name is Vinay.


In [6]:

response3 = conversation.invoke(
    {"question": "I have moved 33 times."},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response3.content)

Wow, that's a lot of moving! Why have you moved so many times?


In [7]:

response4 = conversation.invoke(
    {"question": "If my average moving distance was 100 miles, how many miles took all my moves?"},
    config={"configurable": {"session_id": "user_session_1"}}
)
print(response4.content)

To find the total distance you have moved, you can multiply the average moving distance by the number of times you have moved.

Total distance = Average moving distance x Number of times moved
Total distance = 100 miles x 33 times
Total distance = 3300 miles

So, all your moves combined covered a total distance of 3300 miles. That's quite a journey!


## Conversation Buffer Window Memory
Similar to the previous one, but **you can limit the number of conversational exchanges stored in memory**. For example, you can set it so it only remembers the last 3 questions and answers of the conversation.

In [11]:
from langchain_core.messages import trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. Define the base conversation prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, context-aware chatbot assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 2. Replicate a strict 'k=3' window using trim_messages
# Setting token_counter=len treats every message entry as exactly 1 unit.
# max_tokens=6 ensures we only retain the latest 6 messages (3 human + 3 AI turns).
trimmer = trim_messages(
    max_tokens=6,
    strategy="last",
    token_counter=len,
    start_on="human",       # Ensures the trimmed window never accidentally cuts off an AI response first
    include_system=True     # Prevents the trimmer from accidentally deleting your System prompt instruction
)

# 3. Build an executable LCEL pipeline with inline trimming
# The prompt history field is filtered dynamically right before hitting the LLM model
chain = (
    {
        "history": lambda x: trimmer.invoke(x["history"]),
        "input": lambda x: x["input"]
    }
    | prompt 
    | llm
)

# 4. Bind the pipeline to normal session state tracking
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation_window = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)


C:\Users\DulamVinayGopal\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
config = {"configurable": {"session_id": "chat_session_abc"}}

# Turn 1
print(conversation_window.invoke({"input": "My favorite color is Blue."}, config).content)
# Turn 2
print(conversation_window.invoke({"input": "I work as a software engineer."}, config).content)
# Turn 3
print(conversation_window.invoke({"input": "I love eating pizza."}, config).content)
# Turn 4
print(conversation_window.invoke({"input": "I have a cat named Whiskers."}, config).content)

# Turn 5: Test the memory window limits (k=3)!
# It will remember the cat and the pizza, but it will have forgotten the color Blue (Turn 1).
response = conversation_window.invoke({"input": "What is my favorite color?"}, config)
print(response.content) 


Blue is a beautiful color! It is often associated with calmness, serenity, and stability. Is there anything specific you'd like to know or discuss about the color blue?
That's great! As a software engineer, you play a crucial role in developing and maintaining software applications. Is there anything specific you would like to discuss or any help you need related to your work as a software engineer? Feel free to ask!
Pizza is a popular and delicious choice! There are so many different toppings and varieties to choose from. Do you have a favorite type of pizza or any special toppings you like to add?
That's adorable! Cats make wonderful companions. Do you enjoy spending time with Whiskers and playing with them? If you have any questions or need advice about caring for your cat, feel free to ask!
As an AI chat assistant, I don't have access to personal information about you unless you specifically share it with me during our conversation. Would you like to tell me your favorite color so 

# Chat Message History
* Save list of chat messages and then fetch them all.

In [13]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [15]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# Initialize the modern in-memory history tracker
history = InMemoryChatMessageHistory()

# Add messages exactly as before
history.add_user_message("hi!")
history.add_ai_message("whats up?")


In [16]:
history.messages

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='whats up?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]